# `friend` and Overloading

Two powerful C++ features:
- **`friend`** lets trusted outside code access private members of a class.
- **Overloading** lets you define what operators and function names mean for your own types.

Together they make user-defined types feel natural to work with — you can write `v1 + v2` or `std::cout << myObj` just as you would for built-in types.

## Function Overloading

C++ allows multiple functions with the **same name** as long as their **parameter lists differ**. The compiler picks the right version based on the argument types at the call site.

This is called **overload resolution**.

In [ ]:
#include <iostream>
#include <string>

int add(int a, int b) {
    std::cout << "int add called" << std::endl;
    return a + b;
}

double add(double a, double b) {
    std::cout << "double add called" << std::endl;
    return a + b;
}

std::string add(std::string a, std::string b) {
    std::cout << "string add called" << std::endl;
    return a + b;
}

std::cout << add(3, 4)              << std::endl;
std::cout << add(1.5, 2.5)          << std::endl;
std::cout << add(std::string("Hello, "), std::string("42!")) << std::endl;

**Exercise 1:** Write three overloads of `printInfo`:
- `void printInfo(int n)` — prints `"Int: <n>"`.
- `void printInfo(double d)` — prints `"Double: <d>"`.
- `void printInfo(const std::string &s)` — prints `"String: <s>"`.

Call all three and verify the correct version is selected.

In [ ]:
// Your code here

## Overloading Rules

Functions are distinguished by **parameter types and count**. The return type alone is **not enough** to distinguish overloads.

### This does NOT compile:

```cpp
int    getValue();    // error!
double getValue();    // same name, same params -- ambiguous
```

```
error: functions that differ only in their return type cannot be overloaded
```

### This is fine:

```cpp
int process(int x);              // different param type
int process(double x);           // OK
int process(int x, int y);       // different param count -- OK
```

## Default Arguments

Default arguments let callers omit trailing parameters:

```cpp
void drawShape(int sides, int colour = 0, int size = 1);
```

Rules:
- Default arguments must be the **rightmost** parameters.
- Once a parameter has a default, all parameters to its right must also have defaults.
- Beware **ambiguity** with overloading: `drawShape(6)` could match both `drawShape(int)` and `drawShape(int, int=0, int=1)` if both exist.

In [ ]:
#include <iostream>

void drawShape(int sides, int colour = 0, int size = 1) {
    std::cout << "sides=" << sides
              << " colour=" << colour
              << " size=" << size << std::endl;
}

drawShape(6);          // colour and size use defaults
drawShape(4, 2);       // size uses default
drawShape(3, 1, 5);    // all specified

## Operator Overloading

Operators are just functions with special names:

| Expression | Is equivalent to |
|---|---|
| `a + b` | `a.operator+(b)` or `operator+(a, b)` |
| `a == b` | `a.operator==(b)` or `operator==(a, b)` |
| `std::cout << a` | `operator<<(std::cout, a)` |

We will use `Vector2D` as our example class throughout this section.

In [ ]:
#include <iostream>

class Vector2D {
public:
    Vector2D(double x, double y) : _x(x), _y(y) {}
    Vector2D()                   : _x(0), _y(0) {}

    double getX() const { return _x; }
    double getY() const { return _y; }

    // operator+ as a member function
    Vector2D operator+(const Vector2D &other) const {
        return Vector2D(_x + other._x, _y + other._y);
    }

    void print() const {
        std::cout << "(" << _x << ", " << _y << ")";
    }

private:
    double _x;
    double _y;
};

Vector2D v1(1.0, 2.0);
Vector2D v2(3.0, 4.0);
Vector2D v3 = v1 + v2;   // calls v1.operator+(v2)

std::cout << "v1 = "; v1.print(); std::cout << std::endl;
std::cout << "v2 = "; v2.print(); std::cout << std::endl;
std::cout << "v1 + v2 = "; v3.print(); std::cout << std::endl;

## Arithmetic Operator Overloading

We can extend `Vector2D` with subtraction and scalar multiplication.

In [ ]:
#include <iostream>

// Extended Vector2D with more operators
class Vec2 {
public:
    Vec2(double x, double y) : _x(x), _y(y) {}
    Vec2()                   : _x(0), _y(0) {}

    double getX() const { return _x; }
    double getY() const { return _y; }

    Vec2 operator+(const Vec2 &o) const { return Vec2(_x + o._x, _y + o._y); }
    Vec2 operator-(const Vec2 &o) const { return Vec2(_x - o._x, _y - o._y); }
    Vec2 operator*(double scalar) const { return Vec2(_x * scalar, _y * scalar); }

    Vec2 &operator+=(const Vec2 &o) { _x += o._x; _y += o._y; return *this; }
    Vec2 &operator-=(const Vec2 &o) { _x -= o._x; _y -= o._y; return *this; }

    bool operator==(const Vec2 &o) const { return _x == o._x && _y == o._y; }
    bool operator!=(const Vec2 &o) const { return !(*this == o); }

    void print() const {
        std::cout << "(" << _x << ", " << _y << ")";
    }

private:
    double _x, _y;

    // Grant operator<< access to private members
    friend std::ostream &operator<<(std::ostream &os, const Vec2 &v);
};

Vec2 a(1, 2);
Vec2 b(3, 4);

std::cout << "a = ";       a.print(); std::cout << std::endl;
std::cout << "b = ";       b.print(); std::cout << std::endl;
std::cout << "a - b = ";  (a - b).print(); std::cout << std::endl;
std::cout << "a * 3 = ";  (a * 3).print(); std::cout << std::endl;

**Exercise 2:** Using the `Vec2` class defined above, test `operator+=` and `operator-=`:
1. Create two `Vec2` objects.
2. Use `+=` to accumulate the second into the first.
3. Print before and after to verify.

In [ ]:
// Your code here
// (Vec2 is already defined in the cell above — just use it)

## Comparison Operator Overloading

`operator==` and `operator!=` are already defined inside `Vec2` above. Let's test them.

In [ ]:
#include <iostream>

Vec2 p(2.0, 3.0);
Vec2 q(2.0, 3.0);
Vec2 r(9.0, 0.0);

std::cout << "p == q: " << (p == q ? "true" : "false") << std::endl;   // true
std::cout << "p == r: " << (p == r ? "true" : "false") << std::endl;   // false
std::cout << "p != r: " << (p != r ? "true" : "false") << std::endl;   // true

## The `<<` Operator (Stream Insertion)

`operator<<` for output streams **must be a free function** (not a member), because the left-hand side is `std::ostream` — a class you don't own:

```cpp
std::ostream &operator<<(std::ostream &os, const Vec2 &v);
//            ^^^^^^^^^^^                              ^^^^
//            left side (ostream)        right side (your type)
```

The function returns `std::ostream &` to allow chaining:
```cpp
std::cout << v1 << " and " << v2 << std::endl;
```

In [ ]:
#include <iostream>

// Free function -- defined outside the class
// (Vec2 already declared friend for this function above)
std::ostream &operator<<(std::ostream &os, const Vec2 &v) {
    os << "(" << v._x << ", " << v._y << ")";
    return os;   // enable chaining
}

Vec2 u(5.0, 6.0);
Vec2 w(7.0, 8.0);

// Now we can use << directly
std::cout << u << std::endl;
std::cout << "u + w = " << (u + w) << std::endl;
std::cout << "u: " << u << "  w: " << w << std::endl;   // chained

## `friend` Functions

In the code above, `operator<<` accesses `v._x` and `v._y` — private members. To allow this, we declared it as a **`friend`** inside the class:

```cpp
class Vec2 {
    // ...
    friend std::ostream &operator<<(std::ostream &os, const Vec2 &v);
};
```

A `friend` declaration inside a class grants that external function (or class) full access to all private and protected members. The friend function is **not** a member — it is an external function with special access.

**Exercise 3:** The `operator<<` for `Vec2` is already implemented above using the `friend` declaration. Verify it works by:
1. Creating a `Vec2` named `myVec` with values of your choice.
2. Using `std::cout << myVec << std::endl;` to print it.
3. Printing `myVec * 2.5` using `<<` in the same expression.

In [ ]:
// Your code here

## `friend` Classes

A whole class can be declared a friend, giving it full access to the host class's private members.

```cpp
class Engine {
    friend class Car;   // Car can access Engine's private members
    // ...
};
```

Use this sparingly — it tightly couples two classes. Prefer exposing only what is needed through public methods.

In [ ]:
#include <iostream>

class Engine {
public:
    Engine(int hp) : _horsepower(hp), _running(false) {}

    friend class Car;   // Car gets private access

private:
    int  _horsepower;
    bool _running;
};

class Car {
public:
    Car(const char *model, int hp) : _model(model), _engine(hp) {}

    void start() {
        _engine._running = true;   // directly access private member of Engine
        std::cout << _model << " started (" << _engine._horsepower << " hp)" << std::endl;
    }

    void stop() {
        _engine._running = false;
        std::cout << _model << " stopped" << std::endl;
    }

    bool isRunning() const { return _engine._running; }

private:
    const char *_model;
    Engine      _engine;
};

Car myCar("42-Racer", 300);
myCar.start();
std::cout << "Running: " << (myCar.isRunning() ? "yes" : "no") << std::endl;
myCar.stop();

## The `>>` Operator (Stream Extraction)

Just like `operator<<`, the input operator must be a **free function**. It takes a non-const reference to `std::istream` and a non-const reference to your type (because it modifies it).

```cpp
std::istream &operator>>(std::istream &is, Vec2 &v);
```

If it needs private access, declare it as a `friend` inside the class.

In [ ]:
#include <iostream>
#include <sstream>   // std::istringstream

// We need a version of Vec2 with >> support
class Vec2IO {
public:
    Vec2IO(double x, double y) : _x(x), _y(y) {}
    Vec2IO()                   : _x(0), _y(0) {}

    friend std::ostream &operator<<(std::ostream &os, const Vec2IO &v);
    friend std::istream &operator>>(std::istream &is,       Vec2IO &v);

private:
    double _x, _y;
};

std::ostream &operator<<(std::ostream &os, const Vec2IO &v) {
    os << "(" << v._x << ", " << v._y << ")";
    return os;
}

// Reads two doubles: "3.0 4.5"
std::istream &operator>>(std::istream &is, Vec2IO &v) {
    is >> v._x >> v._y;
    return is;
}

Vec2IO readVec;
std::istringstream input("7.0 8.5");   // simulate user input
input >> readVec;
std::cout << "Read: " << readVec << std::endl;

## Final Exercise

**Exercise 4:** Create a class `Fraction` that represents a rational number (numerator / denominator, both `int`).

Requirements:
- Constructor `Fraction(int num, int den)` — validate that `den != 0` (print an error and set to 0/1 if invalid). Call `simplify()` in the constructor.
- Private helper `void simplify()` that reduces the fraction using GCD. Ensure the denominator is always positive.
- `operator+`, `operator-`, `operator*`, `operator/` — return new `Fraction` objects.
- `operator==`, `operator!=`.
- `friend std::ostream &operator<<(std::ostream &os, const Fraction &f)` — prints as `"n/d"`.

Test: create fractions, do arithmetic, print results.

Hint for GCD (Euclidean algorithm):
```cpp
int gcd(int a, int b) { return b == 0 ? a : gcd(b, a % b); }
```

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### `explicit` constructors

Without `explicit`, a single-argument constructor can be used for **implicit conversion**, which is often surprising:

```cpp
class MyString {
public:
    MyString(int size);   // without explicit
};

void print(const MyString &s);
print(42);   // compiles! Implicitly creates MyString(42) -- usually NOT what you want
```

Adding `explicit` prevents this:

```cpp
explicit MyString(int size);
print(42);             // error: no implicit conversion
print(MyString(42));   // OK: explicit construction
```

### Deleted operators

You can explicitly **forbid** an operation using `= delete`:

```cpp
Fraction &operator=(const Fraction &) = delete;   // disables assignment
```

This is clearer than making the function private and not implementing it (the C++98 idiom).

In [ ]:
#include <iostream>
#include <string>

class Wrapper {
public:
    // explicit: prevents MyWrapper w = 42; (implicit conversion)
    explicit Wrapper(int value) : _value(value) {
        std::cout << "Wrapper(" << _value << ") constructed" << std::endl;
    }

    // Delete copy assignment -- this object cannot be assigned to another
    Wrapper &operator=(const Wrapper &) = delete;

    int getValue() const { return _value; }

private:
    int _value;
};

void printWrapper(const Wrapper &w) {
    std::cout << "Wrapper value: " << w.getValue() << std::endl;
}

Wrapper w1(10);             // OK: explicit construction
// Wrapper w2 = 20;         // ERROR: implicit conversion prevented by explicit
// w1 = Wrapper(30);        // ERROR: operator= is deleted

printWrapper(w1);           // OK: w1 passed by const ref
// printWrapper(5);          // ERROR: implicit conversion prevented
printWrapper(Wrapper(5));   // OK: explicit construction at call site

std::cout << "explicit and = delete working correctly" << std::endl;